In [1]:
"""
Precision Agriculture: Soil Suitability Classification
Implementation of Find-S and Candidate Elimination Algorithms.
Author: Ashwini Vishal Hebbali
"""

class CandidateElimination:
    def __init__(self, num_attributes):
        self.num_attributes = num_attributes
        # S: Most specific hypothesis (initialized to null)
        self.S = ['0'] * num_attributes
        # G: Most general hypothesis (initialized to ?)
        self.G = [['?'] * num_attributes]

    def is_consistent(self, hypothesis, instance):
        """Check if a hypothesis covers an instance."""
        for i in range(self.num_attributes):
            if hypothesis[i] != '?' and hypothesis[i] != instance[i]:
                return False
        return True

    def learn(self, instance, label):
        """Update S and G boundaries based on a new training example."""
        if label.lower() == 'yes':
            # 1. Update S boundary
            for i in range(self.num_attributes):
                if self.S[i] == '0':
                    self.S[i] = instance[i]
                elif self.S[i] != instance[i]:
                    self.S[i] = '?'

            # 2. Prune G boundary (must be consistent with the positive instance)
            self.G = [g for g in self.G if self.is_consistent(g, instance)]

        else:
            # 1. Update G boundary
            new_G = []
            for g in self.G:
                if not self.is_consistent(g, instance):
                    # g is already consistent with the negative instance (doesn't cover it)
                    new_G.append(g)
                else:
                    # g covers the negative instance; must specialize it
                    for i in range(self.num_attributes):
                        if g[i] == '?':
                            # We don't know the full attribute domain here,
                            # so we use placeholders to demonstrate logic
                            # In real usage, you'd iterate through all other possible values
                            for val in ['AlternativeValue']:
                                if val != instance[i]:
                                    specialized_h = list(g)
                                    specialized_h[i] = val
                                    # Ensure specialization is still more general than S
                                    if self.is_consistent(specialized_h, self.S):
                                        new_G.append(specialized_h)
            self.G = new_G

        # Final check: If G becomes empty, the Version Space has collapsed
        if not self.G:
            return self.S, [], True
        return self.S, self.G, False

def run_agriculture_simulation():
    # Attributes: [SoilType, Moisture, PreviousCrop, Drainage, PesticideHistory, BioContent]
    attributes = ["SoilType", "Moisture", "PrevCrop", "Drainage", "Pesticide", "BioContent"]

    # 1. SUCCESSFUL LEARNING CASE
    print("--- Scenario 1: Consistent Agricultural Data ---")
    model = CandidateElimination(len(attributes))
    data = [
        (['Clay', 'High', 'Legumes', 'Good', 'None', 'High'], 'Yes'),
        (['Sandy', 'High', 'Legumes', 'Good', 'None', 'High'], 'Yes'),
        (['Clay', 'Low', 'Wheat', 'Poor', 'High', 'Low'], 'No')
    ]

    for inst, lbl in data:
        s, g, collapsed = model.learn(inst, lbl)
        print(f"Input: {inst} -> Label: {lbl}")
        print(f"S: {s}")
        print(f"G: {g}\n")

    # 2. FAILURE CASE: SENSOR NOISE (Contradictory Labels)
    print("\n--- Scenario 2: Version Space Collapse (Sensor Noise) ---")
    noise_model = CandidateElimination(len(attributes))
    noisy_data = [
        (['Loam', 'Medium', 'Corn', 'Good', 'None', 'High'], 'Yes'),
        (['Loam', 'Medium', 'Corn', 'Good', 'None', 'High'], 'No') # IDENTICAL DATA, CONTRARY LABEL
    ]

    for inst, lbl in noisy_data:
        s, g, collapsed = noise_model.learn(inst, lbl)
        print(f"Input: {inst} -> Label: {lbl}")
        if collapsed:
            print("!!! VERSION SPACE COLLAPSED: Inconsistent data detected. !!!")
            break
        print(f"S: {s}")
        print(f"G: {g}\n")

if __name__ == "__main__":
    run_agriculture_simulation()

--- Scenario 1: Consistent Agricultural Data ---
Input: ['Clay', 'High', 'Legumes', 'Good', 'None', 'High'] -> Label: Yes
S: ['Clay', 'High', 'Legumes', 'Good', 'None', 'High']
G: [['?', '?', '?', '?', '?', '?']]

Input: ['Sandy', 'High', 'Legumes', 'Good', 'None', 'High'] -> Label: Yes
S: ['?', 'High', 'Legumes', 'Good', 'None', 'High']
G: [['?', '?', '?', '?', '?', '?']]

Input: ['Clay', 'Low', 'Wheat', 'Poor', 'High', 'Low'] -> Label: No
S: ['?', 'High', 'Legumes', 'Good', 'None', 'High']
G: []


--- Scenario 2: Version Space Collapse (Sensor Noise) ---
Input: ['Loam', 'Medium', 'Corn', 'Good', 'None', 'High'] -> Label: Yes
S: ['Loam', 'Medium', 'Corn', 'Good', 'None', 'High']
G: [['?', '?', '?', '?', '?', '?']]

Input: ['Loam', 'Medium', 'Corn', 'Good', 'None', 'High'] -> Label: No
!!! VERSION SPACE COLLAPSED: Inconsistent data detected. !!!
